In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

In [2]:
scats_data = pd.ExcelFile('./datasets/Scats Data October 2006.xls')
scats_site_data = pd.ExcelFile('./datasets/SCATSSiteListingSpreadsheet_VicRoads.xlsx')

print(scats_data.sheet_names)
print(scats_site_data.sheet_names)

['Notes', 'Data', 'Summary Of Data']
['Data Infomation', 'SCATS Site Numbers']


In [ ]:
# scats_data = pd.read_excel('./datasets/Scats Data October 2006.xls', sheet_name='Data')
# scats_site_data = pd.read_excel('./datasets/SCATSSiteListingSpreadsheet_VicRoads.xlsx', sheet_name='SCATS Site Numbers')

# # save the dataframes to csv files
# scats_data.to_csv('./datasets/scats_data.csv', index=False)
# scats_site_data.to_csv('./datasets/scats_site_data.csv', index=False)

In [39]:
scats_data = pd.read_csv('./datasets/scats_data.csv')
scats_site_data = pd.read_csv('./datasets/scats_site_data.csv')
traffic_count_data = pd.read_csv('./datasets/Traffic_Count_Locations_with_LONG_LAT.csv')

In [41]:
print(scats_data.isnull().sum())
print(scats_site_data.duplicated().sum())

SCATS Number    0
Location        0
CD_MELWAY       0
NB_LATITUDE     0
NB_LONGITUDE    0
               ..
V91             0
V92             0
V93             0
V94             0
V95             0
Length: 106, dtype: int64
289


In [42]:
# remove duplicates from scats_site_data
scats_data = scats_data.drop_duplicates()
print(scats_data.duplicated().sum())

0


In [44]:
print(scats_data.head())

   SCATS Number                         Location CD_MELWAY  NB_LATITUDE  \
0           970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
1           970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
2           970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
3           970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
4           970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   

   NB_LONGITUDE  HF VicRoads Internal  VR Internal Stat  VR Internal Loc  \
0     145.09159                   249               182                1   
1     145.09159                   249               182                1   
2     145.09159                   249               182                1   
3     145.09159                   249               182                1   
4     145.09159                   249               182                1   

   NB_TYPE_SURVEY              Date  ...  V86  V87  V88  V89  V90  V91  V92  \
0            

In [25]:
print(scats_site_data.head())

   Site Number           Location Description Site Type Directory  \
0          964  ABBOTTS/CLELANDS DEVELOPMENTS       INT    Melway   
1          968           ABBOTTS/GAINE/MONASH       INT    Melway   
2          972               ABBOTTS/NATIONAL       INT    Melway   
3          983              ABBOTTS/REMINGTON       INT    Melway   
4         1053             ABBOTTSFORD/HAINES       INT    Melway   

  Map reference   
0         095G08  
1         095K08  
2          95J08  
3         095G08  
4         2A-G07  


In [26]:
print(traffic_count_data.head())

            X          Y   FID  OBJECTID  TFM_ID  \
0  144.250614 -36.779313  7001      7301    7656   
1  145.356779 -37.835309  7002      7302   29406   
2  144.988844 -37.824629  7003      7303   22676   
3  144.932442 -37.803783  7004      7304   27902   
4  145.030601 -37.660502  7005      7305   10935   

                                TFM_DESC    TFM_TYP_DE MOVEMENT_T  \
0                CALDER HWY NE OF OAK ST  INTERSECTION  All Moves   
1  MT DANDENONG RD S BD SE OF UPALONG RD  INTERSECTION  All Moves   
2              SWAN ST W BD E OF PUNT RD  INTERSECTION  All Moves   
3        DYNON RD W BD E OF RADCLIFFE ST  INTERSECTION  All Moves   
4               DALTON RD N of CHILDS RD  INTERSECTION  All Moves   

                            SITE_DESC  ROAD_NBR            DECLARED_R  \
0                 CALDER HWY & OAK ST      2530        CALDER HIGHWAY   
1    MT DANDENONG RD SE OF UPALONG RD      4991  MOUNT DANDENONG ROAD   
2  PUNT RD LEFT TURN TO SWAN ST OD:12      2080      

In [27]:
print(scats_data.columns)

Index(['SCATS Number', 'Location', 'CD_MELWAY', 'NB_LATITUDE', 'NB_LONGITUDE',
       'HF VicRoads Internal', 'VR Internal Stat', 'VR Internal Loc',
       'NB_TYPE_SURVEY', 'Date',
       ...
       'V86', 'V87', 'V88', 'V89', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95'],
      dtype='object', length=106)


In [46]:
id_vars = scats_data.columns[:10].tolist()

# Step 2: Melt the data into long format
scats_data = pd.melt(scats_data, 
                  id_vars=id_vars, 
                  var_name='Time', 
                  value_name='Volume')

# Step 3: Sort by location, then date, then time
scats_data = scats_data.sort_values(by=['Location', 'Date', 'Time']).reset_index(drop=True)

In [29]:
# Clean both keys
scats_data['CD_MELWAY'] = scats_data['CD_MELWAY'].str.strip().str.upper().str.replace(' ', '').str.replace('-', '')
scats_site_data['Map reference'] = scats_site_data['Map reference '].str.strip().str.upper().str.replace(' ', '').str.replace('-', '')

# Get only one Site Type per Map Reference (e.g. first)
scats_site_lookup = scats_site_data.drop_duplicates(subset='Map reference')

# Rename column to match
scats_site_lookup = scats_site_lookup.rename(columns={'Map reference': 'CD_MELWAY'})

# Now merge: this will NOT create duplicates
scats_data = scats_data.merge(
    scats_site_lookup[['CD_MELWAY', 'Site Type']],
    on='CD_MELWAY',
    how='left'
)
print(scats_data['Site Type'].isnull().sum())
print(scats_data['Site Type'].unique())
print(scats_data['Site Type'].value_counts())
print(scats_data.duplicated().sum())

21888
['INT' 'POS' nan]
Site Type
INT    251136
POS    129408
Name: count, dtype: int64
0


In [30]:
most_common_type = scats_data['Site Type'].mode()[0]
scats_data['Site Type'] = scats_data['Site Type'].fillna(most_common_type)

print(scats_data.isnull().sum())

SCATS Number            0
Location                0
CD_MELWAY               0
NB_LATITUDE             0
NB_LONGITUDE            0
HF VicRoads Internal    0
VR Internal Stat        0
VR Internal Loc         0
NB_TYPE_SURVEY          0
Date                    0
Time                    0
Volume                  0
Site Type               0
dtype: int64


In [31]:
print(scats_data.head())

   SCATS Number                   Location CD_MELWAY  NB_LATITUDE  \
0          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
1          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
2          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
3          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
4          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   

   NB_LONGITUDE  HF VicRoads Internal  VR Internal Stat  VR Internal Loc  \
0           0.0                  4397               653                1   
1           0.0                  4397               653                1   
2           0.0                  4397               653                1   
3           0.0                  4397               653                1   
4           0.0                  4397               653                1   

   NB_TYPE_SURVEY              Date Time  Volume Site Type  
0               1  01/10/2006 00:15  V00      44       INT  
1     

In [32]:
scats_data['Date'] = pd.to_datetime(scats_data['Date'], dayfirst=True)
scats_data['Weekday'] = scats_data['Date'].dt.day_name()
print("Shape after adding Weekday:", scats_data.shape)
print("Duplicates after Weekday:", scats_data.duplicated().sum())

Shape after adding Weekday: (402432, 14)
Duplicates after Weekday: 0


In [ ]:
scats_data['Date'] = pd.to_datetime(scats_data['Date'])

# Sort by group and date (important before diff)
scats_data = scats_data.sort_values(by=['Location', 'Date'])

# Calculate day gap within each location group
scats_data['day_gap'] = scats_data.groupby(['Location'])['Date'].diff().dt.days.fillna(0).astype(int)
print(scats_data.head())
print(scats_data.isnull().sum())

   SCATS Number                   Location CD_MELWAY  NB_LATITUDE  \
0          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
1          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
2          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
3          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   
4          4266  AUBURN_RD N of BURWOOD_RD    045F11          0.0   

   NB_LONGITUDE  HF VicRoads Internal  VR Internal Stat  VR Internal Loc  \
0           0.0                  4397               653                1   
1           0.0                  4397               653                1   
2           0.0                  4397               653                1   
3           0.0                  4397               653                1   
4           0.0                  4397               653                1   

   NB_TYPE_SURVEY                Date Time  Volume Site Type Weekday  day_gap  
0               1 2006-10-01 00:15:00  V00      

In [ ]:
# drop the columns scats number location cd_melway and other internal_fields to avoid noise
scats_data = scats_data.drop(columns=['SCATS Number', 'CD_MELWAY', 'HF VicRoads Internal', 'VR Internal Stat', 'VR Internal Loc', 'NB_TYPE_SURVEY'])

# rename the columns to be more descriptive
# scats_data = scats_data.rename(columns={'NB_LATITUDE': 'Latitude', 'NB_LONGITUDE': 'Longitude'})
print(scats_data.head())

                    Location  Latitude  Longitude                Date Time  \
0  AUBURN_RD N of BURWOOD_RD       0.0        0.0 2006-10-01 00:15:00  V00   
1  AUBURN_RD N of BURWOOD_RD       0.0        0.0 2006-10-01 00:15:00  V01   
2  AUBURN_RD N of BURWOOD_RD       0.0        0.0 2006-10-01 00:15:00  V02   
3  AUBURN_RD N of BURWOOD_RD       0.0        0.0 2006-10-01 00:15:00  V03   
4  AUBURN_RD N of BURWOOD_RD       0.0        0.0 2006-10-01 00:15:00  V04   

   Volume Site Type Weekday  day_gap  
0      44       INT  Sunday        0  
1      53       INT  Sunday        0  
2      36       INT  Sunday        0  
3      35       INT  Sunday        0  
4      37       INT  Sunday        0  


In [18]:
print(scats_data.isnull().sum())
print(scats_data.isna().sum())
print(scats_data.duplicated().sum())

Location     0
Latitude     0
Longitude    0
Date         0
Time         0
Volume       0
Site Type    0
Weekday      0
day_gap      0
dtype: int64
Location     0
Latitude     0
Longitude    0
Date         0
Time         0
Volume       0
Site Type    0
Weekday      0
day_gap      0
dtype: int64
0


In [19]:
scats_data['Date'] = pd.to_datetime(scats_data['Date'], dayfirst=True).dt.date
print(scats_data.head())

                    Location  Latitude  Longitude        Date Time  Volume  \
0  AUBURN_RD N of BURWOOD_RD       0.0        0.0  2006-10-01  V00      44   
1  AUBURN_RD N of BURWOOD_RD       0.0        0.0  2006-10-01  V01      53   
2  AUBURN_RD N of BURWOOD_RD       0.0        0.0  2006-10-01  V02      36   
3  AUBURN_RD N of BURWOOD_RD       0.0        0.0  2006-10-01  V03      35   
4  AUBURN_RD N of BURWOOD_RD       0.0        0.0  2006-10-01  V04      37   

  Site Type Weekday  day_gap  
0       INT  Sunday        0  
1       INT  Sunday        0  
2       INT  Sunday        0  
3       INT  Sunday        0  
4       INT  Sunday        0  


In [ ]:
# Keep only the needed columns (rename as needed)
columns_to_keep = ['Location', 'Longitude', 'Latitude']
df = scats_data[columns_to_keep]

# Drop duplicates based on 'location' (keep first occurrence)
df_unique = df.drop_duplicates(subset='Location', keep='first')

# Export to new CSV
df_unique.to_csv("unique_locations.csv", index=False)

print("✅ Exported unique rows to 'unique_locations.csv'")

✅ Exported unique rows to 'unique_locations.csv'


In [ ]:
import pandas as pd

# Load the uploaded file
df = pd.read_csv("datasets/unique_locations.csv")

# Extract unique place-like names from 'Location' field using regex patterns
# We'll extract named places such as "Burwood", "Warringal", etc. (commonly appearing at the end or after 'of')
import re

places = []

for loc in df['Location']:
    match = re.match(r"(.+?)\s([NSEW]) of\s(.+)", loc, re.IGNORECASE)
    if match:
        places.append(match.group(3).strip())
    else:
        # Also include standalone locations (those not directional)
        places.append(loc.strip())

unique_places = sorted(set([place.upper() for place in places]))

# Create a DataFrame with unique places
unique_places_df = pd.DataFrame(unique_places, columns=['Place'])

# Save to CSV
unique_places_df.to_csv("unique_places.csv", index=False)